In [0]:
CREATE OR REPLACE PROCEDURE clinicalforge.metadata.sp_ProvisionNewCustomer (
    -- Customer Parameters
    p_CustomerCode STRING,
    p_CustomerName STRING,
    p_SubscriptionTier STRING,
    
    -- Product Parameters
    p_ProductCode STRING,
    p_ProductName STRING,
    
    -- ConnectionDetails Parameters
    p_TargetPlatform STRING,
    p_HostServer STRING,
    p_DatabaseName STRING,
    p_UserName STRING,
    p_Password STRING,
    
    -- EventHubDetails Parameters
    p_NamespaceName STRING,
    p_TopicName STRING,
    p_ConnSecretStringEH STRING
)
LANGUAGE SQL
SQL SECURITY INVOKER
BEGIN
    -- Variables to hold dynamic parent table IDs
    DECLARE v_CustomerID INT;
    DECLARE v_ProductID INT;
    DECLARE v_CustomerProductID INT;
    
    --- Variable to Event Hub - boshosp-eventhub-json : /mnt/lakehouse/silver/unified_patient_telemetry
    DECLARE v_EventHubID INT;
    DECLARE VARIABLE v_ConsumerGroup STRING=concat(p_CustomerCode, '-eh-json');
    DECLARE VARIABLE v_TargetDeltaTablePath STRING=CONCAT(p_CustomerCode, '/mnt/lakehouse/silver/unified_patient_telemetry');


    --- Variable to Connection Details - boshosp-connection-json
    DECLARE v_ConnectionID INT;
    DECLARE VARIABLE V_ConnectionName STRING=concat(p_CustomerCode, '-scope');
    DECLARE VARIABLE V_SecretKeyName STRING=CONCAT(p_CustomerCode, '-connection-json');
   
    
    -- Base tracking offsets for bulk multi-row insert generation
    DECLARE v_BaseTableID INT;
    DECLARE v_BaseWarehouseTableID INT;
    -- =========================================================================
    -- 1. PRODUCT RESOLUTION (Find existing or generate new one)
    -- =========================================================================
    set v_ProductID = (SELECT MAX(ProductID) FROM clinicalforge.metadata.Products WHERE ProductCode = p_ProductCode and ProductName =p_ProductName);
    
    IF v_ProductID IS NULL THEN
        set v_ProductID = COALESCE((SELECT MAX(ProductID) FROM clinicalforge.metadata.Products), 0) + 1;
        INSERT INTO clinicalforge.metadata.Products (ProductID, ProductCode, ProductName, ProductDescription, IsActive)
        VALUES (v_ProductID, p_ProductCode, p_ProductName, 'Auto-provisioned pipeline asset', true);
    END IF;

    -- =========================================================================
    -- 2. CORE PARENT ENTITIES CONFIGURATION
    -- =========================================================================
    --1. Generate Customer ID & Insert  -Customers
    set v_CustomerID = COALESCE((SELECT MAX(CustomerID) FROM clinicalforge.metadata.Customers), 0) + 1;
    INSERT INTO clinicalforge.metadata.Customers (CustomerID, CustomerCode, CustomerName, SubscriptionTier, IsActive, CreatedDateTime)
    VALUES (v_CustomerID, p_CustomerCode, p_CustomerName, p_SubscriptionTier, true, current_timestamp());

    --2. Generate CustomerProduct Relationship Bridge ID & Insert - CustomerProduct
    set v_CustomerProductID = COALESCE((SELECT MAX(CustomerProductID) FROM clinicalforge.metadata.CustomerProduct), 0) + 1;
    INSERT INTO clinicalforge.metadata.CustomerProduct (CustomerProductID, CustomerID, ProductID)
    VALUES (v_CustomerProductID, v_CustomerID, v_ProductID);

    --3. Generate Data Connection ID & Insert - ConnectionDetails
    set v_ConnectionID = COALESCE((SELECT MAX(ConnectionID) FROM clinicalforge.metadata.ConnectionDetails), 0) + 1;
    INSERT INTO clinicalforge.metadata.ConnectionDetails (ConnectionID, CustomerProductID, ConnectionName, TargetPlatform, HostServer, DatabaseName, SecretKeyName, IsActive, UserName, Password)
    VALUES (v_ConnectionID, v_CustomerProductID, V_ConnectionName, p_TargetPlatform, p_HostServer, p_DatabaseName, V_SecretKeyName, true, p_UserName, p_Password);

    --4. Generate Streaming EventHub Pipeline ID & Insert - EventHubDetails
    set v_EventHubID = COALESCE((SELECT MAX(EventHubID) FROM clinicalforge.metadata.EventHubDetails), 0) + 1;
    INSERT INTO clinicalforge.metadata.EventHubDetails (EventHubID, CustomerProductID, NamespaceName, TopicName, ConsumerGroup, PartitionCount, ConnectionStringEH, TargetDeltaTablePath)
    VALUES (v_EventHubID, v_CustomerProductID, p_NamespaceName, p_TopicName, v_ConsumerGroup, 4, p_ConnSecretStringEH, v_TargetDeltaTablePath);

    -- =========================================================================
    -- 3. BULK MULTI-RECORD CLONING (Set-Based Execution)
    -- =========================================================================
    
    -- Capture current maximum table ID baseline offsets
    set v_BaseTableID = COALESCE((SELECT MAX(TableID) FROM clinicalforge.metadata.TablesList), 0);
    set v_BaseWarehouseTableID = COALESCE((SELECT MAX(WarehouseTableID) FROM clinicalforge.metadata.CustomerWarehouseTables), 0);

    -- Bulk clone multiple source layout registries for the new customer mapping
    INSERT INTO clinicalforge.metadata.TablesList (TableID, CustomerProductID, SourceConnectionID, SourceSchema, SourceTableName, WarehouseSchema, WarehouseTableName, ExtractionType, WatermarkColumn, LastExtractWatermark, DatabricksNotebookPath, IsActive)
    SELECT 
        v_BaseTableID + ROW_NUMBER() OVER(ORDER BY TableID) as TableID,
        v_CustomerProductID,
        v_ConnectionID,
        SourceSchema,
        SourceTableName,
        WarehouseSchema,
        WarehouseTableName,
        ExtractionType,
        WatermarkColumn,
        CAST('1900-01-01 00:00:00' AS TIMESTAMP),
        DatabricksNotebookPath,
        true
    FROM clinicalforge.metadata.TablesList
    QUALIFY ROW_NUMBER() OVER (PARTITION BY SourceSchema, SourceTableName ORDER BY TableID DESC) = 1;

    -- Bulk clone multiple downstream data warehouse destination registries
    INSERT INTO clinicalforge.metadata.CustomerWarehouseTables (WarehouseTableID, CustomerProductID, TargetSchema, WarehouseTableName, LoadMethod, WatermarkColumn, IsActive, CreatedDateTime, SourceTableName, SourceTableID)
    SELECT 
        v_BaseWarehouseTableID + ROW_NUMBER() OVER(ORDER BY WarehouseTableID) as WarehouseTableID,
        v_CustomerProductID,
        TargetSchema,
        WarehouseTableName,
        LoadMethod,
        WatermarkColumn,
        true,
        current_timestamp(),
        SourceTableName,
        SourceTableID
    FROM clinicalforge.metadata.CustomerWarehouseTables
    QUALIFY ROW_NUMBER() OVER (PARTITION BY TargetSchema, WarehouseTableName ORDER BY WarehouseTableID DESC) = 1;

END;
